# K-coco-stats：CenterNet 中心点碰撞复算（ka-verify-coco-center-collision）

- 主卡：claim-centernetpoints-center-collision-rare-on-coco。预注册判定带：EXP/README.md（先于运行）。
- 口径：R=4，中心=bbox 中心，主口径 floor（官方 decode 惯例），round 作敏感性对照；完整性门 118,287 图 / 860,001 物体。
- 数据：awsaf49/coco-2017-dataset → coco2017/annotations/instances_train2017.json（470MB，只需标注）。

In [ ]:
import json, os, time, itertools, collections

RES = {'kernel': 'ka-verify-coco-center-collision', 'started': time.strftime('%F %T'),
       'git_note': 'preregister commit 见 EXP/README（先于 push）'}
CAND = ['coco2017/annotations/instances_train2017.json', 'annotations/instances_train2017.json']
PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'instances_train2017.json' in files:
        PATH = os.path.join(root, 'instances_train2017.json'); dirs[:] = []
        break
assert PATH, 'ANNOTATIONS_NOT_FOUND: 镜像内未找到 instances_train2017.json'
RES['annotations_path'] = PATH
print('loading', PATH, flush=True)
with open(PATH) as f:
    ann = json.load(f)
imgs = ann['images']; anns = ann['annotations']
RES['n_images'] = len(imgs); RES['n_objects_raw'] = len(anns)
idset = set(im['id'] for im in imgs)
objs = [a for a in anns if a['image_id'] in idset]
RES['n_objects_all'] = len(objs)
RES['n_objects_noncrowd'] = sum(1 for a in objs if not a.get('iscrowd'))
print('images:', len(imgs), '| objects(all, paper口径含 crowd):', len(objs), '| non-crowd:', RES['n_objects_noncrowd'], flush=True)

def collisions(R, rounding):
    cells = collections.Counter()
    areas = {}
    for a in objs:
        x1, y1, w, h = a['bbox']
        cx = (x1 + x1 + w) / 2.0; cy = (y1 + y1 + h) / 2.0
        v = cx / R if rounding == 'floor' else round(cx / R)
        u = cy / R if rounding == 'floor' else round(cy / R)
        if rounding == 'floor':
            v, u = int(v), int(u)
        cells[(a['image_id'], a['category_id'], v, u)] += 1   # v2 修复：碰撞只能发生在同图内（v1 漏 image_id 分组，跨图假对 3.16M 留痕）
        areas[a['id']] = w * h
    pairs = sum(n * (n - 1) // 2 for n in cells.values() if n > 1)
    return pairs, cells, areas

R = 4
pairs_floor, cells_floor, areas = collisions(R, 'floor')
pairs_round, _, _ = collisions(R, 'round')
RES['R'] = R
RES['pairs_floor'] = pairs_floor
RES['pairs_floor_v1_global_grouping_defect'] = 'v1 漏 image_id 分组得 3159187（跨图假对），数字留痕于 RESULTS.md；判定口径不变（614±20%）'
RES['pairs_round'] = pairs_round
RES['ratio_floor'] = pairs_floor / max(1, len(objs))
RES['paper'] = {'pairs': 614, 'objects': 860001, 'R': 4}
import math
in_band = 491 <= pairs_floor <= 737 and pairs_floor / max(1, len(objs)) < 0.001
comp_ok = abs(len(imgs) - 118287) <= 118 and abs(len(objs) - 860001) <= 860
RES['completeness_gate'] = {'images': comp_ok or abs(len(imgs) - 118287) <= 118,
                            'objects_860001': abs(len(objs) - 860001) <= 860}
if not RES['completeness_gate']['objects_860001']:
    RES['verdict'] = 'GATE_FAIL: 物体数不符论文 Appendix C 口径，停止判读照实记录'
elif in_band:
    RES['verdict'] = 'SUPPORT: floor 口径 614±20% 带内且 <0.1%'
elif 491 <= pairs_round <= 737:
    RES['verdict'] = 'ROUNDING_SENSITIVE: floor 出带 round 入带，双报留主人'
elif pairs_floor > 2 * 614 or pairs_floor * 2 < 614:
    RES['verdict'] = 'FALSIFIER-2 DIRECTION HIT: floor 偏离 614 超 2 倍'
else:
    RES['verdict'] = 'OUT_OF_BAND: 记录，不定性'
# 面积分桶（观察，不新增主张）
buckets = {'small(<32^2)': 0, 'medium(32^2-96^2)': 0, 'large(>96^2)': 0, 'bucket_total': 0}
for (img, cat, v, u), n in cells_floor.items():
    if n < 2:
        continue
    k = n * (n - 1) // 2
    buckets['bucket_total'] += k
RES['bucket_note'] = '分桶需要 pair 级 bbox 面积，改为记录各桶物体占比与总体对数，pair 级分桶留收割后人工'
RES['objects_by_area'] = buckets
RES['objects_by_area'] = {
    'small(<32^2)': sum(1 for a in objs if a['bbox'][2] * a['bbox'][3] < 32 ** 2),
    'medium(32^2-96^2)': sum(1 for a in objs if 32 ** 2 <= a['bbox'][2] * a['bbox'][3] < 96 ** 2),
    'large(>96^2)': sum(1 for a in objs if a['bbox'][2] * a['bbox'][3] >= 96 ** 2)}
# 确定性断言：重算一遍
p2, _, _ = collisions(R, 'floor')
RES['deterministic_rerun_equal'] = (p2 == pairs_floor)
RES['finished'] = time.strftime('%F %T')
os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/metrics.json', 'w') as f:
    json.dump(RES, f, ensure_ascii=False, indent=1)
print(json.dumps({k: RES[k] for k in ['n_images', 'n_objects_all', 'pairs_floor', 'pairs_round',
                                      'ratio_floor', 'verdict', 'deterministic_rerun_equal']}, ensure_ascii=False))
print('=== CONCLUSION ===', RES['verdict'])